# B02 — Final cohort, frozen split, dan cache citra

**Jalankan di Colab CPU terlebih dahulu.** Notebook ini membawa salinan metadata dari
`B01_preparation_report.zip` yang sudah diperiksa, sehingga tidak perlu mengunggah laporan itu lagi.
Citra tetap dibaca dari ZIP dataset Anda di Drive. Tidak ada training pada B02.

| Dataset | Citra dipakai | Negatif | Positif | Grup pasien | Dasar label |
|---|---:|---:|---:|---:|---|
| CBIS-DDSM | 3.103 | 1.728 | 1.375 | 1.566 | CSV Notebook 01, label patologi |
| CMMD | 3.744 | 1.112 | 2.632 | 1.775 | Pasien + sisi DICOM cocok dengan data klinis |
| INbreast strict | 344 | 287 | 57 | 100 token | BI-RADS 1/2 negatif; 5/6 positif |
| MIAS abnormal | 115 | 63 | 52 | 108 | Label lesi B/M pada README full MIAS v1.21 |

**Batas interpretasi:** label INbreast merupakan proxy BI-RADS, bukan diagnosis patologi.
Kolom PatientID DICOM INbreast kosong. Token anonim filename digunakan sebagai grup dan
semua kunjungan bertoken sama ditempatkan pada split yang sama; identitas pasien belum
terverifikasi secara independen. Angka 100 adalah grup token dalam cohort strict, bukan klaim
jumlah pasien resmi database. Semua 410 citra memiliki 108 token.

CMMD: 1.458 citra tanpa label sisi yang cocok dikeluarkan, tidak dilabeli benign.
24 citra benign pada kelompok D2 berasal dari 12 pasien yang juga memiliki sisi malignant.
MIAS: 207 citra normal dikeluarkan dari tugas benign versus malignant. `mdb144rx` memiliki
dua lesi B dan M, sehingga label per citranya positif; seluruh baris lanjutan README ikut dibaca.
README halaman 2 menyatakan film ganjil/kiri dan genap/kanan berikutnya berasal dari orang sama.

Split CBIS dari B01 dipertahankan. Tiga dataset lain dibagi sekitar 70/15/15 berdasarkan
grup pasien, seed split 2026, stratifikasi berdasarkan ada/tidaknya citra positif pada pasien.
Label citra tidak diganti oleh label stratifikasi pasien.


## 1. Dependensi dan Drive
Pemasangan decoder DICOM tidak mengunduh dataset. CPU cukup untuk tahap ini.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pydicom>=3,<4', 'pylibjpeg', 'pylibjpeg-libjpeg', 'pylibjpeg-openjpeg',
    'pillow', 'pandas', 'scikit-learn'])
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
RESEARCH = Path('/content/drive/MyDrive/Research')
DATASET_ARCHIVES = {
    'cbis_ddsm': RESEARCH/'datasets/cbis_ddsm_awsaf.zip',
    'cmmd': RESEARCH/'datasets/TheChineseMammographyDatabase.zip',
    'inbreast': RESEARCH/'datasets/inbreast_2012.zip',
    'mias': RESEARCH/'datasets/miasdbv1.21.zip',
}
DATASETS_TO_PREPARE = ['cbis_ddsm','cmmd','inbreast','mias']
OUTPUT_ROOT = RESEARCH/'outputs/benchmark_4datasets/prepared_v1'
LOCAL_ARCHIVE_CACHE = Path('/content/benchmark_b02_archive_cache')
IMAGE_SIZE = 224  # Screening awal; pertahankan sama untuk seluruh 8 model.
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)


## 2. Metadata sumber dan fungsi audit
Cell payload di bawah berisi metadata, bukan gambar. Hash memastikan metadata tertanam utuh.
Untuk CMMD/INbreast/MIAS, fingerprint daftar anggota ZIP harus cocok dengan B01.


In [ ]:
import base64, io, zipfile, hashlib
B01_PAYLOAD = '...'
report_bytes = base64.b64decode(B01_PAYLOAD)
assert hashlib.sha256(report_bytes).hexdigest() == 'b9668070270ca0e67e71273c860a577800c5b2230a62cdeb9b17dff0b91ae116'
SOURCE_REPORT = zipfile.ZipFile(io.BytesIO(report_bytes))


In [ ]:
import io, re, json, math, hashlib, zipfile, shutil, os, time
from pathlib import Path, PurePosixPath
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SPLIT_SEED = 2026
PIPELINE_VERSION = 'benchmark_v1_224_minmax_letterbox'

def sha(data):
    return hashlib.sha256(data).hexdigest()

def csv_read(z, name):
    return pd.read_csv(z.open(name), keep_default_na=False, dtype=str)

def assign_split(frame):
    frame = frame.copy()
    patients = frame.groupby('patient_id', sort=True).label.max().reset_index()
    n = len(patients)
    remain, test = train_test_split(patients, test_size=math.ceil(.15*n),
        random_state=SPLIT_SEED, stratify=patients.label)
    train, val = train_test_split(remain, test_size=math.ceil(.15*n),
        random_state=SPLIT_SEED, stratify=remain.label)
    lookup = {p: s for s, d in [('train', train), ('validation', val), ('test', test)]
              for p in d.patient_id}
    frame['split'] = frame.patient_id.map(lookup)
    return frame

def check_manifest(frame):
    required = ['sample_id', 'patient_id', 'label', 'split', 'dataset']
    assert set(required) <= set(frame.columns)
    assert not frame[required].isna().any().any()
    assert frame.patient_id.str.strip().ne('').all()
    assert not frame.sample_id.duplicated().any()
    assert frame.label.isin([0, 1]).all()
    assert set(frame.split) == {'train', 'validation', 'test'}
    assert frame.groupby('patient_id').split.nunique().max() == 1, 'Patient leakage'
    for _, d in frame.groupby('split'):
        assert set(d.label) == {0, 1}, 'Each split must contain both classes'
    return frame.sort_values(['split', 'sample_id']).reset_index(drop=True)

def parse_mias(text):
    # Continuation lines describe additional lesions in the preceding image.
    base = re.compile(r'^\s*(mdb\d{3}[lr][smlx])\s+([FGD])\s+(NORM|CALC|CIRC|SPIC|MISC|ARCH|ASYM)(?:\s+([BM]))?\b')
    cont = re.compile(r'^\s*(CALC|CIRC|SPIC|MISC|ARCH|ASYM)\s+([BM])\b')
    records, current = {}, None
    for line in text.splitlines():
        m = base.match(line)
        if m:
            name, tissue, lesion, severity = m.groups()
            if name in records:
                raise ValueError('Duplicate MIAS README entry: ' + name)
            current = name
            records[name] = {'image_name':name, 'tissue':tissue,
                             'lesions':[lesion], 'severities':[severity] if severity else []}
        elif (m := cont.match(line)):
            if current is None:
                raise ValueError('MIAS continuation without preceding image')
            records[current]['lesions'].append(m[1])
            records[current]['severities'].append(m[2])
    if len(records) != 322:
        raise ValueError(f'MIAS parsing incomplete: {len(records)} / 322 images')
    rows = []
    for name, d in records.items():
        s = d['severities']
        label = 1 if 'M' in s else 0 if 'B' in s else -1
        if label == -1 and d['lesions'] != ['NORM']:
            raise ValueError(f'Unresolved MIAS severity: {name}')
        serial = int(name[3:6])
        rows.append(dict(image_name=name, patient_id=f'MIAS_{(serial+1)//2:03d}',
            label=label, side='L' if serial % 2 else 'R',
            lesion_types='|'.join(d['lesions']), severities='|'.join(s)))
    return pd.DataFrame(rows)

def build_cohorts(z):
    cohorts, excluded = {}, {}
    cbis = pd.concat([csv_read(z, f'cbis_{s}.csv') for s in ['train','validation','test']], ignore_index=True)
    cbis['label'] = pd.to_numeric(cbis.label, errors='raise').astype(int)
    cbis['dataset'] = 'cbis_ddsm'
    cbis['patient_group_source'] = 'CBIS patient_id'
    cbis['label_source'] = 'existing Notebook01 pathology aggregation'
    cohorts['cbis_ddsm'] = check_manifest(cbis)

    cmmd = csv_read(z, 'cmmd_candidate_mapping.csv')
    assert cmmd.status.eq('OK').all()
    assert cmmd.patient_from_path.eq(cmmd.patient_from_header).all()
    assert cmmd.SOPInstanceUID.ne('').all() and not cmmd.SOPInstanceUID.duplicated().any()
    good = cmmd.mapping_status.eq('MAPPED_BY_EXPLICIT_SIDE')
    excluded['cmmd'] = cmmd.loc[~good, ['member','patient_from_path','side','mapping_status']].copy()
    c = cmmd.loc[good].copy()
    c['label'] = pd.to_numeric(c.label, errors='raise').astype(int)
    assert pd.to_numeric(c.label_count).eq(1).all()
    assert c.groupby('clinical_key').size().eq(2).all()
    c['patient_id'] = c.patient_from_path
    c['patient_group_source'] = 'DICOM PatientID checked against path'
    c['label_source'] = 'clinical classification matched by patient + ImageLaterality'
    def decode_view(value):
        codes = {a['value'] for a in json.loads(value)}
        if '399162004' in codes: return 'CC'
        if '399368009' in codes: return 'MLO'
        raise ValueError('CMMD unknown view code: ' + value)
    c['view'] = c.view_codes.map(decode_view)
    assert c.groupby('clinical_key').view.nunique().eq(2).all()
    c['dataset'] = 'cmmd'
    c['sample_id'] = c.member.map(lambda p: sha(('cmmd|' + p).encode()))
    cohorts['cmmd'] = check_manifest(assign_split(c))

    ib = csv_read(z, 'inbreast_candidate_mapping.csv')
    assert ib.status.eq('OK').all() and ib['_merge'].eq('both').all()
    assert not ib.SOPInstanceUID.duplicated().any()
    assert ib.filename_side.eq(ib.Laterality_y).all()
    labels = ib.birads_normalized.map({'1':0, '2':0, '5':1, '6':1})
    excluded['inbreast'] = ib.loc[labels.isna(), ['member','birads_normalized']].copy()
    excluded['inbreast']['exclusion_reason'] = 'BI-RADS 3/4: excluded from strict proxy task'
    i = ib.loc[labels.notna()].copy()
    i['label'] = labels.loc[i.index].astype(int)
    assert pd.to_numeric(i.label_candidate).eq(i.label).all()
    assert i.patient_token.str.fullmatch(r'[0-9a-fA-F]+').all()
    i['patient_id'] = 'IN_' + i.patient_token
    i['patient_group_source'] = 'filename pseudonym; DICOM PatientID unavailable'
    i['label_source'] = 'BI-RADS 1/2 vs 5/6; proxy, not pathology'
    i['dataset'], i['side'], i['view'] = 'inbreast', i.filename_side, i.View
    i['sample_id'] = i.member.map(lambda p: sha(('inbreast|' + p).encode()))
    cohorts['inbreast'] = check_manifest(assign_split(i))

    labels = parse_mias(z.read('mias_00README.txt').decode('utf-8'))
    pgm = csv_read(z, 'mias_pgm_headers.csv')
    pgm['image_name'] = pgm.member.map(lambda p: PurePosixPath(p).stem)
    m = pgm.merge(labels, on='image_name', validate='one_to_one', how='outer', indicator=True)
    assert m['_merge'].eq('both').all()
    assert labels.set_index('image_name').loc['mdb144rx', 'label'] == 1
    excluded['mias'] = m.loc[m.label.eq(-1)].copy()
    excluded['mias']['exclusion_reason'] = 'Normal: excluded from benign versus malignant primary task'
    m = m.loc[m.label.ge(0)].copy()
    m['dataset'] = 'mias'
    m['patient_group_source'] = 'README paired films: odd left, next even right'
    m['label_source'] = 'full MIAS v1.21 README; any malignant lesion => image positive'
    m['sample_id'] = m.member.map(lambda p: sha(('mias|' + p).encode()))
    cohorts['mias'] = check_manifest(assign_split(m))
    return cohorts, excluded

def summarize(cohorts):
    return pd.DataFrame([dict(dataset=k, split=s, images=len(g),
        patient_groups=g.patient_id.nunique(), negative_images=int(g.label.eq(0).sum()),
        positive_images=int(g.label.eq(1).sum()))
        for k, d in cohorts.items() for s, g in d.groupby('split')])

def archive_index(path):
    with zipfile.ZipFile(path) as z:
        rows = [dict(member=x.filename, bytes=x.file_size, compressed_bytes=x.compress_size,
                     crc32=f'{x.CRC:08x}') for x in z.infolist() if not x.is_dir()]
    return sha(json.dumps(rows, sort_keys=True).encode())

def jpeg_key(value):
    parts = str(value).replace('\\', '/').strip('/').split('/')
    positions = [n for n, part in enumerate(parts) if part.lower() in ('jpeg','jpg')]
    if positions:
        parts = parts[positions[-1] + 1:]
    else:
        parts = parts[-2:]  # UID + file, never basename alone
    return '/'.join(parts).lower()

def locate_cbis(frame, archive):
    with zipfile.ZipFile(archive) as z:
        members = [x.filename for x in z.infolist() if not x.is_dir()
                   and x.filename.lower().endswith(('.jpg','.jpeg'))]
    lookup = {}
    for p in members:
        lookup.setdefault(jpeg_key(p), []).append(p)
    matched, errors = [], []
    for p in frame.image_path:
        hits = lookup.get(jpeg_key(p), [])
        matched.append(hits[0] if len(hits)==1 else '')
        if len(hits)!=1: errors.append({'image_path':p,'key':jpeg_key(p),'matches':len(hits)})
    if errors:
        raise RuntimeError('CBIS ZIP path mismatch. ' + json.dumps({
            'total_jpegs':len(members), 'unmatched_or_ambiguous':len(errors),
            'csv_examples':errors[:3], 'zip_examples':members[:5]}, indent=2))
    out = frame.copy(); out['member'] = matched
    if out.member.duplicated().any():
        raise RuntimeError('More than one CBIS record maps to the same JPEG')
    return out

def image_to_uint8(raw, member, size):
    from PIL import Image
    if member.lower().endswith('.dcm'):
        import pydicom
        from pydicom.pixels import apply_modality_lut
        ds = pydicom.dcmread(io.BytesIO(raw))
        pixels = np.asarray(apply_modality_lut(ds.pixel_array, ds))
        invert = str(ds.get('PhotometricInterpretation', '')) == 'MONOCHROME1'
        if str(ds.get('PhotometricInterpretation', '')) not in ('MONOCHROME1','MONOCHROME2'):
            raise ValueError('Unsupported DICOM photometric interpretation')
    else:
        with Image.open(io.BytesIO(raw)) as im:
            pixels = np.asarray(im.convert('L'))
        invert = False
    if pixels.ndim != 2:
        raise ValueError('Expected one 2-D mammogram: ' + member)
    source_pixel_sha = sha(str((pixels.shape, str(pixels.dtype))).encode() + pixels.tobytes())
    a = pixels.astype(np.float32)
    if not np.isfinite(a).all() or a.max() <= a.min():
        raise ValueError('Nonfinite or constant image: ' + member)
    lo, hi = float(a.min()), float(a.max())
    a = (a-lo)/(hi-lo)
    if invert: a = 1-a
    u8 = np.rint(np.clip(a,0,1)*255).astype(np.uint8)
    h, w = u8.shape
    scale = min(size/w, size/h)
    nw, nh = max(1, round(w*scale)), max(1, round(h*scale))
    small = Image.fromarray(u8).resize((nw,nh), Image.Resampling.BILINEAR)
    canvas = Image.new('L',(size,size),0)
    canvas.paste(small, ((size-nw)//2, (size-nh)//2))
    out = io.BytesIO(); canvas.save(out, format='PNG')
    return out.getvalue(), dict(source_pixel_sha256=source_pixel_sha,
        processed_pixel_sha256=sha(np.asarray(canvas).tobytes()),
        original_height=h, original_width=w, pixel_min=lo, pixel_max=hi)

def stream_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024), b''): h.update(b)
    return h.hexdigest()

def atomic_json(path, obj):
    path = Path(path); tmp = path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)

def local_archive(source, expected_index, cache_root):
    source = Path(source)
    if not source.exists(): raise FileNotFoundError(source)
    current = archive_index(source)
    if expected_index and current != expected_index:
        raise RuntimeError('Archive changed since B01: ' + str(source))
    cache_root.mkdir(parents=True, exist_ok=True)
    target = cache_root / (source.stem+'_'+current[:16]+'.zip')
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        if shutil.disk_usage(cache_root).free < source.stat().st_size + 2*1024**3:
            raise RuntimeError('Insufficient local disk for archive + preparation')
        print('Copying archive to local Colab disk:', source.name, flush=True)
        tmp = target.with_suffix('.partial')
        with source.open('rb') as src, tmp.open('wb') as dst:
            shutil.copyfileobj(src, dst, 8*1024*1024)
        os.replace(tmp,target)
    if archive_index(target) != current:
        raise RuntimeError('Local archive index mismatch')
    return target, current

def prepare_dataset(key, frame, source, expected_index, output_root, cache_root, size=224):
    # The fingerprint includes exact cohort/split labels and the preprocessing version.
    identity = frame[['sample_id','patient_id','label','split']].sort_values('sample_id').to_csv(index=False)
    cohort_sha = sha(identity.encode())
    fingerprint = sha((PIPELINE_VERSION+'|'+str(size)+'|'+cohort_sha).encode())[:16]
    dest = output_root/key/fingerprint
    dest.mkdir(parents=True, exist_ok=True)
    ready = dest/'READY.json'
    if ready.exists():
        data = json.loads(ready.read_text())
        if archive_index(source) != data['archive_index_sha256']:
            raise RuntimeError('Source archive changed since the ready cache was created')
        for filename, digest in data['file_sha256'].items():
            if stream_sha(dest/filename) != digest:
                raise RuntimeError('Existing cache integrity failure: '+filename)
        print(key, 'verified existing complete cache')
        return dict(status='READY', path=str(dest), **data)
    archive, index_sha = local_archive(source, expected_index, cache_root)
    frame = locate_cbis(frame, archive) if key=='cbis_ddsm' else frame.copy()
    tmp_zip = cache_root/(key+'_'+fingerprint+'_pixels.partial.zip')
    records = []
    with zipfile.ZipFile(archive) as src, zipfile.ZipFile(tmp_zip,'w',compression=zipfile.ZIP_STORED) as dst:
        for j, row in enumerate(frame.to_dict('records')):
            raw = src.read(row['member'])  # ZIP CRC is verified by read()
            png, audit = image_to_uint8(raw, row['member'], size)
            cache_member = row['sample_id']+'.png'
            dst.writestr(cache_member,png)
            records.append(dict(**row, cache_member=cache_member, source_file_sha256=sha(raw), **audit))
            if (j+1)%100==0 or j+1==len(frame): print(key,j+1,'/',len(frame),flush=True)
    result = pd.DataFrame(records)
    result.to_csv(dest/'all.csv',index=False)
    # Detect identical decoded pixels and identical model inputs across splits.
    for col in ['source_pixel_sha256','processed_pixel_sha256']:
        groups = result.groupby(col).agg(splits=('split','nunique'),labels=('label','nunique'),n=('sample_id','size'))
        bad = groups.index[(groups.splits>1)|(groups.labels>1)]
        if len(bad):
            result.loc[result[col].isin(bad)].to_csv(dest/(col+'_conflicts.csv'),index=False)
            raise RuntimeError(f'{key}: identical images cross splits or conflict in label. Review conflict CSV; no ready marker written.')
    for split, d in result.groupby('split'): d.to_csv(dest/(split+'.csv'),index=False)
    staged = dest/'images.zip.partial'
    shutil.copyfile(tmp_zip,staged); os.replace(staged,dest/'images.zip')
    info = dict(dataset=key, pipeline_version=PIPELINE_VERSION, image_size=size,
        cohort_sha256=cohort_sha, archive_index_sha256=index_sha, source_archive=str(source),
        images=len(result), patient_groups=int(result.patient_id.nunique()), split_seed=SPLIT_SEED,
        grouping_caveat='INbreast uses unverified filename pseudonym' if key=='inbreast' else '',
        duplicates_checked='exact source decoded pixels and exact resized pixels; near duplicates not assessed',
        file_sha256={f:stream_sha(dest/f) for f in ['all.csv','train.csv','validation.csv','test.csv','images.zip']})
    atomic_json(ready,info)
    return dict(status='READY', path=str(dest), **info)


## 3. Bekukan cohort dan periksa pemisahan pasien
Tidak ada pemilihan threshold atau model menggunakan test. Test hanya diringkas jumlahnya.


In [ ]:
COHORTS, EXCLUDED = build_cohorts(SOURCE_REPORT)
B01_REPORT = json.loads(SOURCE_REPORT.read('B01_report.json'))
SPLIT_SUMMARY = summarize(COHORTS)
display(SPLIT_SUMMARY)
SPLIT_SUMMARY.to_csv(OUTPUT_ROOT/'split_summary.csv',index=False)
for key, table in EXCLUDED.items():
    table.to_csv(OUTPUT_ROOT/(key+'_excluded.csv'),index=False)
for key, table in COHORTS.items():
    table.to_csv(OUTPUT_ROOT/(key+'_frozen_metadata.csv'),index=False)
print('Split pasien/grup konsisten; kedua kelas hadir pada semua split.')


## 4. Baca piksel dan buat cache

1. ZIP disalin ke disk lokal Colab agar pembacaan tidak berulang melalui Drive.
2. JPEG/PGM dibaca grayscale. DICOM didekode dengan modality LUT bila ada dan
   MONOCHROME1 dibalik. VOI/window khusus vendor tidak diterapkan.
3. Semua citra dinormalisasi min–max per citra ke 0–255, kemudian diperkecil
   proporsional ke 224×224 dengan padding hitam. Tidak ada ROI crop atau CLAHE.
4. PNG cache, hash piksel, label, dan identitas split disimpan. Ini adalah preprocessing baru,
   bukan klaim kesetaraan piksel dengan notebook lama atau DICOM resmi versus Kaggle.
5. Duplikat piksel persis lintas split atau label yang bertentangan menghentikan dataset
   terkait. Near-duplicate belum diperiksa. Semua kasus yang gagal dicatat; dataset lain tetap diproses.

**224×224 merupakan konfigurasi screening, bukan resolusi yang sudah terbukti optimal untuk lesi kecil.**
Jika resolusi/protokol diubah, perbandingan final perlu menggunakan protokol yang sama untuk semua model.
Cache tidak menggunakan statistik validation/test untuk fitting normalisasi.


In [ ]:
registry_path=OUTPUT_ROOT/'registry.json'
REGISTRY_READY=json.loads(registry_path.read_text()) if registry_path.exists() else {}
for key in DATASETS_TO_PREPARE:
    print('Preparing',key,flush=True)
    expected = None if key=='cbis_ddsm' else B01_REPORT['datasets'][key]['index_sha256']
    try:
        info=prepare_dataset(key,COHORTS[key],DATASET_ARCHIVES[key],expected,
            OUTPUT_ROOT,LOCAL_ARCHIVE_CACHE,IMAGE_SIZE)
        REGISTRY_READY[key]=info
    except Exception as exc:
        REGISTRY_READY[key]={'status':'FAILED','error':str(exc),'dataset':key}
        print(key,'FAILED:',str(exc),flush=True)
    atomic_json(registry_path,REGISTRY_READY)
display(pd.DataFrame([{'dataset':k,'status':v['status'],'detail':v.get('path',v.get('error',''))}
    for k,v in REGISTRY_READY.items()]))


## 5. Tinjau contoh train dan ekspor laporan
Periksa apakah citra utuh, orientasi masuk akal, dan intensitas tidak kosong.
Contoh hanya diambil dari train. Pemeriksaan visual ini melengkapi audit otomatis.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
for key,entry in REGISTRY_READY.items():
    if entry['status']!='READY': continue
    folder=Path(entry['path'])
    train=pd.read_csv(folder/'train.csv')
    sample=pd.concat([g.sample(min(3,len(g)),random_state=42) for _,g in train.groupby('label')])
    fig,axes=plt.subplots(1,len(sample),figsize=(3*len(sample),4),squeeze=False)
    with zipfile.ZipFile(folder/'images.zip') as z:
        for ax,row in zip(axes[0],sample.itertuples()):
            ax.imshow(Image.open(io.BytesIO(z.read(row.cache_member))),cmap='gray',vmin=0,vmax=255)
            ax.set_title(f'{key}; label {row.label}');ax.axis('off')
    plt.tight_layout();plt.show()


In [ ]:
REPORT_ZIP=OUTPUT_ROOT/'B02_ready_report.zip'
with zipfile.ZipFile(REPORT_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for f in [registry_path,OUTPUT_ROOT/'split_summary.csv']:
        z.write(f,f.name)
    for f in OUTPUT_ROOT.glob('*_excluded.csv'): z.write(f,f.name)
    for key,entry in REGISTRY_READY.items():
        if entry['status']=='READY':
            folder=Path(entry['path'])
            for name in ['READY.json','train.csv','validation.csv','test.csv']:
                z.write(folder/name,key+'/'+name)
print('Report:',REPORT_ZIP)
print('Dataset berstatus READY dapat langsung digunakan di B03.')
print('Jika CBIS gagal, kirim B02_ready_report.zip; jangan mengubah label/path berdasarkan tebakan.')
from google.colab import files
files.download(str(REPORT_ZIP))
